# Corrected NLP Sentiment Analysis - Amazon All Beauty Reviews, 2015+

This notebook replaces the synthetic fallback and raw-data mismatch in the
earlier NLP template. It analyzes the canonical full cleaned dataset:
`amazon_beauty_full_2015plus.csv.gz`.

**Why the full dataset?** The main NLP research question does not require
price, so retaining unpriced products avoids price-selection bias. The
clustering module separately applies the same sentiment definitions to the
priced subset.

**Corrections:**

- real reviews only; no synthetic fallback
- inclusive 2015 cutoff inherited and revalidated
- `asin` used consistently as the product key
- no incompatible `parent_asin` merge
- review and product exports come from the same cleaned population
- VADER sentiment, rating-sentiment discrepancy, hidden dissatisfaction,
  and early/later review measures retained from the original methodology

In [ ]:
from pathlib import Path
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from nltk.sentiment import SentimentIntensityAnalyzer

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 120)

EARLY_WINDOW = 5
DATA_PATH = Path("amazon_beauty_full_2015plus.csv.gz")
OUTPUT_DIR = Path(".")
REVIEW_OUTPUT = OUTPUT_DIR / "sentiment_scores_per_review_2015plus.csv.gz"
PRODUCT_OUTPUT = OUTPUT_DIR / "sentiment_scores_per_product_2015plus.csv"

print(f"Input: {DATA_PATH}")

## 1. Load and validate the full cleaned 2015+ review population

In [ ]:
usecols = [
    "user_id", "asin", "rating", "title", "text",
    "helpful_vote", "verified_purchase", "review_date",
    "review_year", "review_length",
]
reviews = pd.read_csv(
    DATA_PATH,
    usecols=usecols,
    parse_dates=["review_date"],
    low_memory=False,
)

reviews["rating"] = pd.to_numeric(reviews["rating"], errors="coerce")
reviews["helpful_vote"] = pd.to_numeric(
    reviews["helpful_vote"], errors="coerce"
).fillna(0)
reviews["verified_purchase"] = (
    reviews["verified_purchase"].astype(str).str.lower()
    .map({"true": True, "false": False})
)
reviews = reviews.dropna(
    subset=["user_id", "asin", "rating", "review_date"]
).copy()

year_check = reviews["review_year"].min() >= 2015
date_check = reviews["review_date"].min() >= pd.Timestamp("2015-01-01")
duplicate_check = not reviews.duplicated(["user_id", "asin"]).any()
print(f"[{'PASS' if year_check else 'WARNING'}] Earliest review year is 2015 or later.")
print(f"[{'PASS' if date_check else 'WARNING'}] Earliest review date is 2015-01-01 or later.")
print(f"[{'PASS' if duplicate_check else 'WARNING'}] User-product review pairs are unique.")

print(f"Reviews: {len(reviews):,}")
print(f"Products (ASINs): {reviews['asin'].nunique():,}")
print(f"Users: {reviews['user_id'].nunique():,}")
print(
    f"Date range: {reviews['review_date'].min()} to "
    f"{reviews['review_date'].max()}"
)

## 2. Clean text and calculate VADER sentiment

In [ ]:
def clean_text(value):
    if pd.isna(value):
        return ""
    value = str(value).lower()
    value = re.sub(r"http\S+|www\.\S+", " ", value)
    value = re.sub(r"[^a-z0-9\s']", " ", value)
    return re.sub(r"\s+", " ", value).strip()

sia = SentimentIntensityAnalyzer()
reviews["clean_text"] = reviews["text"].map(clean_text)
reviews["sentiment_compound"] = reviews["clean_text"].map(
    lambda text: sia.polarity_scores(text)["compound"] if text else 0.0
)

reviews["sentiment_label"] = np.select(
    [
        reviews["sentiment_compound"].ge(0.05),
        reviews["sentiment_compound"].le(-0.05),
    ],
    ["positive", "negative"],
    default="neutral",
)

print(reviews["sentiment_label"].value_counts())
reviews[
    ["rating", "text", "sentiment_compound", "sentiment_label"]
].head()

## 3. Quantify rating-sentiment discrepancy

Star ratings are mapped from 1-5 onto `[-1, 1]`. The discrepancy measure is:

`sentiment_compound - scaled_rating`

Negative values mean the review text is less positive than the star rating
implies. Hidden dissatisfaction is conservatively defined as a 4-5-star
review whose text has negative VADER compound sentiment.

In [ ]:
reviews["scaled_rating"] = (reviews["rating"] - 3.0) / 2.0
reviews["discrepancy"] = (
    reviews["sentiment_compound"] - reviews["scaled_rating"]
)
reviews["hidden_dissatisfaction"] = (
    reviews["rating"].ge(4)
    & reviews["sentiment_compound"].lt(-0.05)
)

high_rating_mask = reviews["rating"].ge(4)
hidden_rate_high_ratings = reviews.loc[
    high_rating_mask, "hidden_dissatisfaction"
].mean()
hidden_rate_all_reviews = reviews["hidden_dissatisfaction"].mean()

print(
    "Hidden dissatisfaction among 4-5-star reviews: "
    f"{hidden_rate_high_ratings:.2%}"
)
print(
    "Hidden dissatisfaction among all reviews: "
    f"{hidden_rate_all_reviews:.2%}"
)

## 4. Identify early versus later reviews within each ASIN

In [ ]:
reviews = reviews.sort_values(["asin", "review_date", "user_id"]).copy()
reviews["review_rank"] = reviews.groupby("asin").cumcount() + 1
reviews["review_stage"] = np.where(
    reviews["review_rank"].le(EARLY_WINDOW), "early", "later"
)

stage_summary = reviews.groupby("review_stage").agg(
    n_reviews=("rating", "size"),
    avg_rating=("rating", "mean"),
    avg_sentiment=("sentiment_compound", "mean"),
    avg_discrepancy=("discrepancy", "mean"),
    pct_hidden_dissatisfaction=("hidden_dissatisfaction", "mean"),
).round(4)
stage_summary

## 5. Diagnostic figures

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

plot_sample = reviews.sample(
    n=min(50000, len(reviews)), random_state=42
)
colors = plot_sample["hidden_dissatisfaction"].map(
    {True: "crimson", False: "steelblue"}
)
axes[0].scatter(
    plot_sample["rating"], plot_sample["sentiment_compound"],
    c=colors, alpha=.18, s=8,
)
axes[0].axhline(0, color="gray", linewidth=.8)
axes[0].set(
    title="Rating vs. text sentiment (sample)",
    xlabel="Star rating",
    ylabel="VADER compound sentiment",
)

reviews.boxplot(
    column="sentiment_compound", by="review_stage",
    ax=axes[1], grid=False,
)
axes[1].set(
    title="Sentiment: early vs. later reviews",
    xlabel="Review stage",
    ylabel="VADER compound sentiment",
)
plt.suptitle("")
plt.tight_layout()
plt.show()

## 6. Export corrected per-review sentiment scores

In [ ]:
review_columns = [
    "user_id", "asin", "review_date", "rating",
    "verified_purchase", "helpful_vote", "review_length",
    "sentiment_compound", "sentiment_label", "scaled_rating",
    "discrepancy", "hidden_dissatisfaction",
    "review_rank", "review_stage",
]
per_review_export = reviews[review_columns].copy()
per_review_export.to_csv(
    REVIEW_OUTPUT,
    index=False,
    compression={"method": "gzip", "compresslevel": 6, "mtime": 0},
    date_format="%Y-%m-%d %H:%M:%S.%f",
)
print(f"Saved: {REVIEW_OUTPUT}")
print(f"Rows: {len(per_review_export):,}")

## 7. Export corrected per-product sentiment summaries

In [ ]:
early_summary = (
    reviews.loc[reviews["review_stage"].eq("early")]
    .groupby("asin")
    .agg(
        early_n_reviews=("rating", "size"),
        early_avg_rating=("rating", "mean"),
        early_avg_sentiment=("sentiment_compound", "mean"),
        early_avg_discrepancy=("discrepancy", "mean"),
    )
)
later_summary = (
    reviews.loc[reviews["review_stage"].eq("later")]
    .groupby("asin")
    .agg(
        later_n_reviews=("rating", "size"),
        later_avg_rating=("rating", "mean"),
        later_avg_sentiment=("sentiment_compound", "mean"),
        later_avg_discrepancy=("discrepancy", "mean"),
    )
)

per_product_export = reviews.groupby("asin").agg(
    n_reviews=("rating", "size"),
    avg_rating=("rating", "mean"),
    rating_std=("rating", "std"),
    avg_sentiment=("sentiment_compound", "mean"),
    sentiment_std=("sentiment_compound", "std"),
    avg_discrepancy=("discrepancy", "mean"),
    pct_hidden_dissatisfaction=("hidden_dissatisfaction", "mean"),
    pct_verified=("verified_purchase", "mean"),
    avg_helpful_votes=("helpful_vote", "mean"),
    first_review=("review_date", "min"),
    last_review=("review_date", "max"),
).join(early_summary).join(later_summary).reset_index()

per_product_export["rating_std"] = (
    per_product_export["rating_std"].fillna(0)
)
per_product_export["sentiment_std"] = (
    per_product_export["sentiment_std"].fillna(0)
)
per_product_export["sentiment_change_later_minus_early"] = (
    per_product_export["later_avg_sentiment"]
    - per_product_export["early_avg_sentiment"]
)
per_product_export["rating_change_later_minus_early"] = (
    per_product_export["later_avg_rating"]
    - per_product_export["early_avg_rating"]
)

per_product_export.to_csv(
    PRODUCT_OUTPUT,
    index=False,
    date_format="%Y-%m-%d %H:%M:%S.%f",
)
print(f"Saved: {PRODUCT_OUTPUT}")
print(f"Products: {len(per_product_export):,}")
per_product_export.head()

## 8. Final quality-control checks

In [ ]:
quality_checks = {
    "Review export has the expected number of rows": len(per_review_export) == len(reviews),
    "Exported user-product review pairs are unique": not per_review_export.duplicated(["user_id", "asin"]).any(),
    "Exported reviews begin on or after 2015-01-01": per_review_export["review_date"].min() >= pd.Timestamp("2015-01-01"),
    "Sentiment compound scores are between -1 and 1": per_review_export["sentiment_compound"].between(-1, 1).all(),
    "Ratings are between 1 and 5": per_review_export["rating"].between(1, 5).all(),
    "Product ASINs are unique in the product export": per_product_export["asin"].is_unique,
    "Product export has one row per reviewed product": len(per_product_export) == reviews["asin"].nunique(),
    "Hidden-dissatisfaction percentages are between 0 and 1": per_product_export["pct_hidden_dissatisfaction"].between(0, 1).all(),
}
for check_name, passed in quality_checks.items():
    print(f"[{'PASS' if passed else 'WARNING'}] {check_name}")

if all(quality_checks.values()):
    print("All quality-control checks passed.")
else:
    print("One or more quality-control checks raised a warning; review the messages above.")
print(
    f"Review output size: {REVIEW_OUTPUT.stat().st_size / 1e6:.1f} MB"
)
print(
    f"Product output size: {PRODUCT_OUTPUT.stat().st_size / 1e6:.1f} MB"
)

## Interpretation caution

VADER is a lexicon-based sentiment model. It is useful for scalable,
reproducible analysis of short consumer reviews but can misread sarcasm,
product-specific terminology, and mixed statements. Hidden dissatisfaction
is therefore an analytical signal, not a definitive classification of an
individual customer's experience.